# Herbicide_Symptomology_Classification_23Mar23

Lead code author: Joe Johnson

**Data Loading**

**Pre-Processing data**

In [ ]:
from tensorflow import keras
import numpy as np
import tensorflow as tf

In [ ]:
img_dim = 200

In [ ]:
val_ds = keras.utils.image_dataset_from_directory(
    'C:/File',
    labels='inferred',
    image_size=(img_dim, img_dim),
    label_mode='int',
    batch_size=16,
    shuffle=True,
    seed=7,
    validation_split = 0.18,
    subset="validation",
    interpolation='bicubic'
)

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    'C:/File',
    labels='inferred',
    image_size=(img_dim, img_dim),
    label_mode='int',
    batch_size=16,
    shuffle=True,
    seed=7,
    validation_split = 0.1891,
    subset="training",
    interpolation='bicubic'
)

In [ ]:
class_names = train_ds.class_names
print(class_names)

In [ ]:
class_weights_dict = {0:1.21786736, 1:0.97713886, 2:1.11429872, 3:4.16687952, 4:0.47182454,
       5:1.01640706, 6:1.86544692, 7:1.96309667, 8:0.53495108}

https://scikit-learn.org/stable/modules/generated/sklearn.utils.class_weight.compute_class_weight.html#

In [ ]:
resize_and_rescale = keras.Sequential([
  #layers.experimental.preprocessing.Resizing(IMG_SIZE, IMG_SIZE),
  keras.layers.experimental.preprocessing.Rescaling(1./255)
])

In [ ]:
data_augmentation = keras.Sequential([
    keras.layers.experimental.preprocessing.RandomFlip("horizontal_and_vertical"),
    #keras.layers.experimental.preprocessing.RandomRotation(0.2),
    #keras.layers.experimental.preprocessing.RandomFlip("vertical"),
])

#train_ds = train_ds.map(lambda x, y: (resize_and_rescale(x, training=True), y))
#https://www.tensorflow.org/tutorials/images/data_augmentation

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def prepare(ds, augment=False):
    # Resize and rescale all datasets.
    ds = ds.map(lambda x, y: (resize_and_rescale(x), y),num_parallel_calls=AUTOTUNE)

    # Use data augmentation only on the training set.
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),num_parallel_calls=AUTOTUNE)

    # Use buffered prefetching on all datasets.
    return ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
train_ds = prepare(train_ds, augment=True)
val_ds = prepare(val_ds)

# Xception model

In [ ]:
model = keras.applications.Xception(weights=None, include_top=False, input_shape=(img_dim, img_dim, 3))
#model = keras.applications.Xception(weights="imagenet", include_top=False, input_shape=(299, 299, 3))

In [ ]:
# Unfreeze the base model
model.trainable = True

In [ ]:
# remove the output layer
#model = keras.Model(inputs=model_Xception.inputs, outputs=model_Xception.layers[-2].output)
model = keras.Model(inputs=model.inputs, outputs=model.output)
#x = model(model1.output, training = True)


https://machinelearningmastery.com/how-to-use-transfer-learning-when-developing-convolutional-neural-network-models/

In [ ]:
#flat1 = keras.layers.Flatten()(model.layers[-1].output)
x = keras.layers.GlobalAveragePooling2D()(model.output)
#flat1 = keras.layers.Flatten()(x)
class1 = keras.layers.Dense(512, activation='relu')(x)
output = keras.layers.Dense(9, activation='softmax')(class1)
# define new model
model = keras.Model(inputs=model.inputs, outputs=output)

In [ ]:
model.summary()

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='sparse_categorical_accuracy',
    verbose=1,
    patience=10,
    mode='max',
    restore_best_weights=True)

In [ ]:
'''
METRICS = [
      keras.metrics.TruePositives(name='tp'),
      keras.metrics.FalsePositives(name='fp'),
      keras.metrics.TrueNegatives(name='tn'),
      keras.metrics.FalseNegatives(name='fn'),
      keras.metrics.BinaryAccuracy(name='accuracy'),
      keras.metrics.Precision(name='precision'),
      keras.metrics.Recall(name='recall'),
      keras.metrics.AUC(name='auc'),
      keras.metrics.AUC(name='prc', curve='PR'), # precision-recall curve
]
'''

In [ ]:
#model1.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3, decay=1e-4), loss='sparse_categorical_crossentropy', metrics=["sparse_categorical_accuracy"])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3, decay=1e-4), loss='sparse_categorical_crossentropy', metrics=["sparse_categorical_accuracy"])

Training the model

In [ ]:
history1 = model.fit(train_ds, epochs=50, batch_size=16, validation_data=val_ds, callbacks=[early_stopping], class_weight=class_weights_dict)
#history = model.fit(x=tr_image_batch, y=tr_label_batch, epochs=2, batch_size=2, validation_split=0.2, callbacks=[early_stopping], class_weight=class_weights_dict, verbose=1)

In [ ]:
#model.save('G:/My Drive/TAMU/Projects/Herbicide_Symtom_Ubaldo/ground_imgs/' + 'xception_30ep_8batchsize')

In [ ]:
import matplotlib.pyplot as plt
loss = history1.history["sparse_categorical_accuracy"]
val_loss = history1.history["val_sparse_categorical_accuracy"]
epochs = range(1, len(loss) + 1)
plt.figure()
plt.plot(epochs, loss, "ro-", label="Training Acc")
plt.plot(epochs, val_loss, "b", label="Validation Acc")
plt.title("Training and validation Accuracy")
plt.legend()
plt.savefig('C:/File' + "tr_val_acc_plot_xception_50ep_16bs_1.jpg")
plt.show()


In [ ]:
loss = history1.history["loss"]
val_loss = history1.history["val_loss"]
epochs = range(1, len(loss) + 1)
plt.figure()
plt.plot(epochs, loss, "ro-", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation Loss")
plt.legend()
plt.savefig('C:/File' + "tr_val_loss_plot_xception_50ep_16bs_1.jpg")
plt.show()

In [ ]:
#import tensorflow as tf
tf.saved_model.save(model,'C:/File' + 'xception_50ep_16bs_1')

In [ ]:
test_ds = keras.utils.image_dataset_from_directory(
    'C:/File',
    labels='inferred',
    image_size=(img_dim, img_dim),
    label_mode='int',
    batch_size=2202,
    shuffle=True,
    seed=7,
    validation_split = 0.18,
    subset="validation",
    interpolation='bicubic'
)

In [ ]:
test_ds = prepare(test_ds)

In [ ]:
ts_image_batch, ts_label_batch = next(iter(test_ds))

In [ ]:
ts_image_batch = np.asarray(ts_image_batch)
ts_label_batch = np.asarray(ts_label_batch)

In [ ]:
test1 = model.evaluate(x=ts_image_batch, y=ts_label_batch)

In [ ]:
preds = model.predict(ts_image_batch)

In [ ]:
y_pred = []
for i in preds:
    y_pred = np.append(y_pred,np.argmax(i))

In [ ]:
y_pred.astype(int)

In [ ]:
ts_label_batch

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt


In [ ]:
cm = confusion_matrix(ts_label_batch, y_pred, labels = [0,1,2,3,4,5,6,7,8])

In [ ]:
#labels = ['24D-1131', 'atz-1383', 'di-1211', 'gly-334', 'isox-2883', 'lib-1355', 'nico-697', 'ntc-680', 'para-2560']
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0,1,2,3,4,5,6,7,8])
disp.plot()
plt.title("0:24D, 1:atz, 2:di, 3:gly, 4:isox, 5:lib, 6:nico, 7:ntc, 8:para")
plt.savefig('C:/File' + "confusion_mat_xception_50ep_16bs_1.jpg")
plt.show()

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(ts_label_batch, y_pred, labels = [0,1,2,3,4,5,6,7,8]))

Preprocessing a single image

In [ ]:
ts_image_batch.shape

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.utils import array_to_img
# importing the random module
import random

i = random.randint(0,10)
plt.axis("off")
plt.imshow(array_to_img(ts_image_batch[i]))

predict single image

In [ ]:
test_img = np.reshape(ts_image_batch[i], (1,img_dim,img_dim,3))
preds = model.predict(test_img)
print(preds)

In [ ]:
np.argmax(preds[0])

Setting up a model that returns the last convolutional output

In [ ]:
last_conv_layer_name = "block14_sepconv2_act"

last_conv_layer = model.get_layer(last_conv_layer_name)
last_conv_layer_model = keras.Model(model.inputs, last_conv_layer.output)

Reapplying the classifier on top of the last convolutional output

In [ ]:
classifier_layer_names = [
    "global_average_pooling2d",
    #"flatten",
    "dense",
    "dense_1",
]
classifier_input = keras.Input(shape=last_conv_layer.output.shape[1:])
x = classifier_input
for layer_name in classifier_layer_names:
    x = model.get_layer(layer_name)(x)
classifier_model = keras.Model(classifier_input, x)

Retrieving the gradients of the top predicted class

In [ ]:
import tensorflow as tf

with tf.GradientTape() as tape:
    last_conv_layer_output = last_conv_layer_model(test_img)
    tape.watch(last_conv_layer_output)
    preds = classifier_model(last_conv_layer_output)
    top_pred_index = tf.argmax(preds[0])
    top_class_channel = preds[:, top_pred_index]

grads = tape.gradient(top_class_channel, last_conv_layer_output)

Gradient pooling and channel-importance weighting

In [ ]:
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2)).numpy()
last_conv_layer_output = last_conv_layer_output.numpy()[0]
for i in range(pooled_grads.shape[-1]):
    last_conv_layer_output[:, :, i] *= pooled_grads[i]
heatmap = np.mean(last_conv_layer_output, axis=-1)

In [ ]:
heatmap

Heatmap post-processing

In [ ]:
heatmap = np.maximum(heatmap, 0)
heatmap /= np.max(heatmap)
plt.matshow(heatmap)

In [ ]:
img = np.uint8(255 * test_img)

Superimposing the heatmap on the original picture

In [ ]:
import matplotlib.cm as cm

#img = keras.utils.load_img(img_path)
img = keras.utils.img_to_array(test_img[0])
img = np.uint8(255 * img)
#img1 = array_to_img(ts_image_batch[1])
heatmap = np.uint8(255 * heatmap)

jet = cm.get_cmap("jet")
jet_colors = jet(np.arange(256))[:, :3]
jet_heatmap = jet_colors[heatmap]

jet_heatmap = keras.utils.array_to_img(jet_heatmap)
jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
jet_heatmap = keras.utils.img_to_array(jet_heatmap)

superimposed_img = jet_heatmap * 0.4 + img
superimposed_img = keras.utils.array_to_img(superimposed_img)

plt.imshow(superimposed_img)

# Interpreting what convnets learn

Visualizing intermediate activations

Instantiating a model that returns layer activations

In [ ]:
from tensorflow.keras import layers

layer_outputs = []
layer_names = []
for layer in model.layers:
    if isinstance(layer, (layers.Conv2D, layers.MaxPooling2D)):
        layer_outputs.append(layer.output)
        layer_names.append(layer.name)
activation_model = keras.Model(inputs=model.input, outputs=layer_outputs)

Using the model to compute layer activations

In [ ]:
activations = activation_model.predict(test_img)

first_layer_activation = activations[0]
print(first_layer_activation.shape)

Visualizing the fifth channel

In [ ]:
import matplotlib.pyplot as plt
plt.matshow(first_layer_activation[0, :, :, 5], cmap="viridis")

Visualizing every channel in every intermediate activation

In [ ]:
images_per_row = 16
for layer_name, layer_activation in zip(layer_names, activations):
    n_features = layer_activation.shape[-1]
    size = layer_activation.shape[1]
    n_cols = n_features // images_per_row
    display_grid = np.zeros(((size + 1) * n_cols ,
                             images_per_row * (size + 1) ))
    for col in range(n_cols):
        for row in range(images_per_row):
            channel_index = col * images_per_row + row
            channel_image = layer_activation[0, :, :, channel_index].copy()
            if channel_image.sum() != 0:
                channel_image -= channel_image.mean()
                channel_image /= channel_image.std()
                channel_image *= 64
                channel_image += 128
            channel_image = np.clip(channel_image, 0, 255).astype("uint8")
            display_grid[
                col * (size + 1): (col + 1) * size + col,
                row * (size + 1) : (row + 1) * size + row] = channel_image
    scale = 1. / size
    plt.figure(figsize=(scale * display_grid.shape[1],
                        scale * display_grid.shape[0]))
    plt.title(layer_name)
    plt.grid(False)
    plt.axis("off")
    plt.imshow(display_grid, aspect="auto", cmap="viridis")

# InceptionResNetV2 model

In [ ]:
model = keras.applications.InceptionResNetV2(weights=None, include_top=False, input_shape=(img_dim, img_dim, 3))

In [ ]:
# Unfreeze the base model
model.trainable = True

In [ ]:
model = keras.Model(inputs=model.inputs, outputs=model.output)

In [ ]:
x = keras.layers.GlobalAveragePooling2D()(model.output)
class1 = keras.layers.Dense(512, activation='relu')(x)
output = keras.layers.Dense(9, activation='softmax')(class1)
# define new model
model = keras.Model(inputs=model.inputs, outputs=output)

In [ ]:
model.summary()

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='sparse_categorical_accuracy',
    verbose=1,
    patience=10,
    mode='max',
    restore_best_weights=True)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3, decay=1e-4), loss='sparse_categorical_crossentropy', metrics=["sparse_categorical_accuracy"])

In [ ]:
history2 = model.fit(train_ds, epochs=50, batch_size=16, validation_data=val_ds, callbacks=[early_stopping], class_weight=class_weights_dict)

In [ ]:
import matplotlib.pyplot as plt
loss = history2.history["sparse_categorical_accuracy"]
val_loss = history2.history["val_sparse_categorical_accuracy"]
epochs = range(1, len(loss) + 1)
plt.figure()
plt.plot(epochs, loss, "ro-", label="Training Acc")
plt.plot(epochs, val_loss, "b", label="Validation Acc")
plt.title("Training and validation Accuracy")
plt.legend()
plt.savefig('C:/File' + "tr_val_acc_plot_InceptionResNetV2_50ep_16bs_1.jpg")
plt.show()

In [ ]:
loss = history2.history["loss"]
val_loss = history2.history["val_loss"]
epochs = range(1, len(loss) + 1)
plt.figure()
plt.plot(epochs, loss, "ro-", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation Loss")
plt.legend()
plt.savefig('C:/File' + "tr_val_loss_plot_InceptionResNetV2_50ep_16bs_1.jpg")
plt.show()

In [ ]:
#import tensorflow as tf
tf.saved_model.save(model,'C:/File' + 'InceptionResNetV2_50ep_16bs_1')

# DenseNet201 model

In [ ]:
model = keras.applications.DenseNet201(weights=None, include_top=False, input_shape=(img_dim, img_dim, 3))

In [ ]:
# Unfreeze the base model
model.trainable = True

In [ ]:
model = keras.Model(inputs=model.inputs, outputs=model.output)

In [ ]:
x = keras.layers.GlobalAveragePooling2D()(model.output)
class1 = keras.layers.Dense(512, activation='relu')(x)
output = keras.layers.Dense(9, activation='softmax')(class1)
# define new model
model = keras.Model(inputs=model.inputs, outputs=output)

In [ ]:
model.summary()

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='sparse_categorical_accuracy',
    verbose=1,
    patience=10,
    mode='max',
    restore_best_weights=True)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3, decay=1e-4), loss='sparse_categorical_crossentropy', metrics=["sparse_categorical_accuracy"])

In [ ]:
history3 = model.fit(train_ds, epochs=50, batch_size=16, validation_data=val_ds, callbacks=[early_stopping], class_weight=class_weights_dict)

In [ ]:
import matplotlib.pyplot as plt
loss = history3.history["sparse_categorical_accuracy"]
val_loss = history3.history["val_sparse_categorical_accuracy"]
epochs = range(1, len(loss) + 1)
plt.figure()
plt.plot(epochs, loss, "ro-", label="Training Acc")
plt.plot(epochs, val_loss, "b", label="Validation Acc")
plt.title("Training and validation Accuracy")
plt.legend()
plt.savefig('C:/File' + "tr_val_acc_plot_DenseNet201_50ep_16bs_1.jpg")
plt.show()

In [ ]:
loss = history3.history["loss"]
val_loss = history3.history["val_loss"]
epochs = range(1, len(loss) + 1)
plt.figure()
plt.plot(epochs, loss, "ro-", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation Loss")
plt.legend()
plt.savefig('C:/File' + "tr_val_loss_plot_DenseNet201_50ep_16bs_1.jpg")
plt.show()

In [ ]:
#import tensorflow as tf
tf.saved_model.save(model,'C:/File' + 'DenseNet201_50ep_16bs_1')

# EfficientNetV2M model

In [ ]:
model = keras.applications.EfficientNetV2M(weights=None, include_top=False, input_shape=(img_dim, img_dim, 3))

In [ ]:
# Unfreeze the base model
model.trainable = True

In [ ]:
model = keras.Model(inputs=model.inputs, outputs=model.output)

In [ ]:
x = keras.layers.GlobalAveragePooling2D()(model.output)
class1 = keras.layers.Dense(512, activation='relu')(x)
output = keras.layers.Dense(9, activation='softmax')(class1)
# define new model
model = keras.Model(inputs=model.inputs, outputs=output)

In [ ]:
model.summary()

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='sparse_categorical_accuracy',
    verbose=1,
    patience=10,
    mode='max',
    restore_best_weights=True)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3, decay=1e-4), loss='sparse_categorical_crossentropy', metrics=["sparse_categorical_accuracy"])

In [ ]:
history4 = model.fit(train_ds, epochs=50, batch_size=16, validation_data=val_ds, callbacks=[early_stopping], class_weight=class_weights_dict)

In [ ]:
import matplotlib.pyplot as plt
loss = history4.history["sparse_categorical_accuracy"]
val_loss = history4.history["val_sparse_categorical_accuracy"]
epochs = range(1, len(loss) + 1)
plt.figure()
plt.plot(epochs, loss, "ro-", label="Training Acc")
plt.plot(epochs, val_loss, "b", label="Validation Acc")
plt.title("Training and validation Accuracy")
plt.legend()
plt.savefig('C:/File' + "tr_val_acc_plot_EfficientNetV2M_50ep_16bs_1.jpg")
plt.show()

In [ ]:
loss = history4.history["loss"]
val_loss = history4.history["val_loss"]
epochs = range(1, len(loss) + 1)
plt.figure()
plt.plot(epochs, loss, "ro-", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation Loss")
plt.legend()
plt.savefig('C:/File' + "tr_val_loss_plot_EfficientNetV2M_50ep_16bs_1.jpg")
plt.show()

In [ ]:
#import tensorflow as tf
tf.saved_model.save(model,'C:/File' + 'EfficientNetV2S_50ep_16bs_1')

# MobileNetV2 model

In [ ]:
model = keras.applications.MobileNetV2(weights=None, include_top=False, input_shape=(img_dim, img_dim, 3))

In [ ]:
# Unfreeze the base model
model.trainable = True

In [ ]:
model = keras.Model(inputs=model.inputs, outputs=model.output)

In [ ]:
x = keras.layers.GlobalAveragePooling2D()(model.output)
class1 = keras.layers.Dense(512, activation='relu')(x)
output = keras.layers.Dense(9, activation='softmax')(class1)
# define new model
model = keras.Model(inputs=model.inputs, outputs=output)

In [ ]:
model.summary()

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='sparse_categorical_accuracy',
    verbose=1,
    patience=10,
    mode='max',
    restore_best_weights=True)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3, decay=1e-4), loss='sparse_categorical_crossentropy', metrics=["sparse_categorical_accuracy"])

In [ ]:
history5 = model.fit(train_ds, epochs=50, batch_size=16, validation_data=val_ds, callbacks=[early_stopping], class_weight=class_weights_dict)

In [ ]:
import matplotlib.pyplot as plt
loss = history5.history["sparse_categorical_accuracy"]
val_loss = history5.history["val_sparse_categorical_accuracy"]
epochs = range(1, len(loss) + 1)
plt.figure()
plt.plot(epochs, loss, "ro-", label="Training Acc")
plt.plot(epochs, val_loss, "b", label="Validation Acc")
plt.title("Training and validation Accuracy")
plt.legend()
plt.savefig('C:/File' + "tr_val_acc_plot_MobileNetV2_50ep_16bs_1.jpg")
plt.show()

In [ ]:
loss = history5.history["loss"]
val_loss = history5.history["val_loss"]
epochs = range(1, len(loss) + 1)
plt.figure()
plt.plot(epochs, loss, "ro-", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation Loss")
plt.legend()
plt.savefig('C:/File' + "tr_val_loss_plot_MobileNetV2_50ep_16bs_1.jpg")
plt.show()

In [ ]:
#import tensorflow as tf
tf.saved_model.save(model,'C:/File' + 'MobileNetV2_50ep_16bs_1')

# ViT model

https://keras.io/examples/vision/image_classification_with_vision_transformer/

pip install -U tensorflow-addons

In [ ]:
import tensorflow_addons as tfa

In [ ]:
num_classes = 9
input_shape = (img_dim, img_dim, 3)

Configure the hyperparameters

In [ ]:
learning_rate = 0.001
weight_decay = 0.0001
batch_size = 16
num_epochs = 50
image_size = img_dim  # We'll resize input images to this size
patch_size = 20  # Size of the patches to be extract from the input images
num_patches = (image_size // patch_size) ** 2
projection_dim = 64
num_heads = 8
transformer_units = [
    projection_dim * 2,
    projection_dim,
]  # Size of the transformer layers
transformer_layers = 8
mlp_head_units = [512] #[256, 128]  # Size of the dense layers of the final classifier

Implement multilayer perceptron (MLP)

In [ ]:
def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = keras.layers.Dense(units, activation=tf.nn.gelu)(x)
        x = keras.layers.Dropout(dropout_rate)(x)
    return x

Implement patch creation as a layer

In [ ]:
class Patches(keras.layers.Layer):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches

Implement the patch encoding layer

In [ ]:
class PatchEncoder(keras.layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.num_patches = num_patches
        self.projection = keras.layers.Dense(units=projection_dim)
        self.position_embedding = keras.layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded

**Build the ViT model**

In [ ]:
def create_vit_classifier():
    inputs = keras.layers.Input(shape=input_shape)
    # Augment data.
    augmented = data_augmentation(inputs)
    # Create patches.
    patches = Patches(patch_size)(augmented)
    # Encode patches.
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    # Create multiple layers of the Transformer block.
    for _ in range(transformer_layers):
        # Layer normalization 1.
        x1 = keras.layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        # Create a multi-head attention layer.
        attention_output = keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=0.1
        )(x1, x1)
        # Skip connection 1.
        x2 = keras.layers.Add()([attention_output, encoded_patches])
        # Layer normalization 2.
        x3 = keras.layers.LayerNormalization(epsilon=1e-6)(x2)
        # MLP.
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=0.1)
        # Skip connection 2.
        encoded_patches = keras.layers.Add()([x3, x2])

    # Create a [batch_size, projection_dim] tensor.
    representation = keras.layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = keras.layers.Flatten()(representation)
    representation = keras.layers.Dropout(0.5)(representation)
    # Add MLP.
    features = mlp(representation, hidden_units=mlp_head_units, dropout_rate=0.5)
    # Classify outputs.
    logits = keras.layers.Dense(num_classes)(features)
    # Create the Keras model.
    model = keras.Model(inputs=inputs, outputs=logits)
    return model

Compile, train, and evaluate the mode

In [ ]:
model = create_vit_classifier()

In [ ]:
model.summary()

In [ ]:
optimizer = tfa.optimizers.AdamW(learning_rate=learning_rate, weight_decay=weight_decay)
model.compile(optimizer=optimizer,
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy"),],)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    verbose=1,
    patience=10,
    mode='max',
    restore_best_weights=True)

In [ ]:
history6 = model.fit(train_ds, epochs=10, batch_size=16, validation_data=val_ds, callbacks=[early_stopping], class_weight=class_weights_dict)

In [ ]:
import matplotlib.pyplot as plt
loss = history6.history["accuracy"]
val_loss = history6.history["val_accuracy"]
epochs = range(1, len(loss) + 1)
plt.figure()
plt.plot(epochs, loss, "ro-", label="Training Acc")
plt.plot(epochs, val_loss, "b", label="Validation Acc")
plt.title("Training and validation Accuracy")
plt.legend()
plt.savefig('C:/File' + "tr_val_acc_plot_ViT_50ep_16bs_1.jpg")
plt.show()

In [ ]:
loss = history6.history["loss"]
val_loss = history6.history["val_loss"]
epochs = range(1, len(loss) + 1)
plt.figure()
plt.plot(epochs, loss, "ro-", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation Loss")
plt.legend()
plt.savefig('C:/File' + "tr_val_loss_plot_ViT_50ep_16bs_1.jpg")
plt.show()

In [ ]:
tf.saved_model.save(model,'C:/File' + 'ViT_50ep_16bs_1')

_____________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________

**Program to resize images to 640x640 pixels**

In [ ]:
import PIL
import os
from PIL import Image

Path of the images to a variable:

In [ ]:
from_path = r'C:/File'

In [ ]:
to_path = r'C:/File'

In [ ]:
for file in os.listdir(from_path):
    f_img = from_path+"/"+file
    img = Image.open(f_img)
    img = img.resize((640,640))
    img.save(to_path+'/'+file)